# 1.环境检查

In [1]:
import torch
print('cuda available:', torch.cuda.is_available())
print('torch', torch.__version__)

cuda available: True
torch 2.13.0+cu126


# 2.数据准备

In [3]:
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
tf = transforms.Compose([
    transforms.ToTensor(),               # PIL → Tensor，值域自动 0~1
    transforms.Normalize((0.5,), (0.5,)) # (x-0.5)/0.5 → 值域 -1~1
])
data = "/home/wsl2/py-learning-log/21-torch/data"
train_ds = datasets.FashionMNIST(data, train=True, download=True, transform=tf)
test_ds  = datasets.FashionMNIST(data, train=False, download=True, transform=tf)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)
print('train batches:', len(train_loader), '| test batches:', len(test_loader))

train batches: 235 | test batches: 40


要点：
- 用 torchvision 直接下载，notebook 自包含（不依赖 d2l 工具）。这个 notebook 的主题是"训练工程"，不在数据上花时间。Normalize((0.5,), (0.5,)) 把像素从 0~1 拉到 -1~1（归一化让梯度更稳）；测试集 shuffle=False——顺序固定，评估结果可复现。

# 3. 小 MLP

In [15]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.net(x)
module = MLP()
print(module)

MLP(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=10, bias=True)
  )
)


要点：
- 线性堆叠的结构用 Sequential 足够，不用手写 forward 里一层层调；需要跳连/多输入时才手写 forward（VAE 那种就要手写）。

为什么一来就展平 nn.Flatten ()
- Fashion‑MNIST 输入是图像张量：`[B, 1, 28, 28]`
  - 维度含义：批次 B，通道 = 1，高 28，宽 28
  - **全连接层 nn.Linear 只接受二维输入 `[B,特征数]`，不能接收 4 维图像张量**。
  - `nn.Linear(784,256)` 的第一个参数要求输入特征是 784，所以必须把 `1×28×28` 像素全部拉直成一维向量 784。 
> CNN 不需要一开始就展平：卷积层可以直接处理 4 维图像`[B,C,H,W]`；MLP（多层感知机）天生只能处理向量，图像必须先拉平。

为什么只激活了一次，中间只写了一个 ReLU<br>
nn.Linear(784, 256),<br>
nn.ReLU(),<br>
nn.Linear(256, 10)<br>
- 网络结构：
  - `784 →Linear→256 →ReLU→256 →Linear→10`
  - 第一个 Linear 输出 256，经过 ReLU 做非线性；
  - **最后一层 Linear (256,10) 后面没有激活**。
- 这个网络实际只有**1 层隐藏层**，所以只有 1 次 ReLU。
- 如果你要更深，就要多组 `Linear+ReLU`
> 每一个隐藏全连接后面都要跟激活；输出层看损失函数决定要不要激活

最后一层全连接为什么不用激活？
- 任务：10 分类，损失函数用 `nn.CrossEntropyLoss()`
- `CrossEntropyLoss = log_softmax + NLLLoss`
- **损失函数内部自带 Softmax 运算**。
- 网络输出直接输出 logits（原始分数，可正可负）交给 loss。
- 如果网络末尾自己再加一层`nn.Softmax(dim=1)`，会重复计算，数值不稳定、收敛变差。
- 两种场景对比：
1. 多分类 + CrossEntropyLoss：输出层**不加激活**，输出 logits ✅
2. 二分类 + BCEWithLogitsLoss：输出层不加 sigmoid ✅
3. 二分类单独用 BCELoss：输出层必须加 Sigmoid
4. VAE 解码器生成 0‑1 图像：输出层加 Sigmoid
> 记住：带 Logits 的 loss，网络输出不要激活。

MLP 和其他网络的区别
- MLP（多层感知机，全连接网络）
  - 输入必须是向量，图像要先展平；
  - 每一个输入像素和隐藏层每一个神经元全部两两相连；
  - 参数量大：784→256 就有 784*256 权重；
  - **没有利用图像空间局部信息**：像素的位置关系完全丢掉，打乱图片像素顺序，MLP 依然输出一样结果。
  - 适合表格数据；图像任务能力弱。
- CNN 卷积网络（LeNet、ResNet）
  - 直接接收 4 维图像`[B,C,H,W]`，**不需要一开始展平**；
  - 使用卷积核，只和局部相邻像素做运算；
  - 参数少，利用图像空间、相邻像素信息；
  - 图像任务主力。
- VAE
  - 编码器把图像压缩到低维隐向量 z；解码器再把 z 还原回图像；
  - 不是单纯分类，是生成 + 表征学习。

# 4.保存函数

In [16]:
def save_checkpoint(path, epoch, model,optimizer, acc):
    torch.save({
        'epoch': epoch,   # 记录当前训练到第几轮；恢复断点时可以从该 epoch 继续往下训练
        'model_state_dict': model.state_dict(),   # 只提取网络可训练参数 (weight、bias)，不是整个 model 对象
        'optimizer_state_dict': optimizer.state_dict(),   # 保存优化器内部状态：学习率、动量、Adam 的一阶 / 二阶矩缓存
        'acc': acc,   # 把验证精度存入断点文件；后续加载时可以知道这个 checkpoint 的性能，方便区分 best /last 权重
    }, path)
    print(f'[save] epoch {epoch}, acc {acc:.4f} -> {path}')

要点（checkpoint 铁律）：
- torch.save(model.state_dict())，绝不 torch.save(model)——后者把整个对象序列化，类定义一改、换台机器就废；前者只存纯权重张量字典，可移植。恢复训练必须连 optimizer 一起存：SGD 的动量、Adam 的 m/v 都是状态，不存 = 续训时优化器状态清零，之前的动量白算了。
- 函数定义：把 Python 对象序列化写到磁盘,保存**完整断点 (checkpoint)**，不是只存模型权重。
- 入参：
  - `path`：保存文件路径，例如 `"best_ckpt.pt"`
  - `epoch`：当前第几轮
  - `model`：网络模型对象
  - `optimizer`：优化器对象
  - `acc`：当前验证集精度，记录下来方便回看

# 5. 训练循环（带保存）

In [22]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(module.parameters(), lr = 0.1)
best_acc = 0.0  # 记录历史最高精度，用来判断要不要保存最优权重
def train_one_epoch(module, loader, loss_fn, optimizer):
    module.train()
    correct = 0    # 统计本 epoch 预测正确的样本总数
    for X, y in loader:
        pred = module(X)
        loss = loss_fn(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        correct += (pred.argmax(1) == y).sum().item()
    return correct / len(loader.dataset)
for epoch in range(6):
    acc = train_one_epoch(module, train_loader, loss_fn, optimizer)
    print(f'epoch {epoch}: train acc {acc:.4f}')
    if acc > best_acc:  # 如果本轮训练集准确率超过历史最好，就调用上面写好的函数保存断点
        best_acc = acc
        save_checkpoint('best_ckpt.pt', epoch, module, optimizer, acc)

epoch 0: train acc 0.8759
[save] epoch 0, acc 0.8759 -> best_ckpt.pt
epoch 1: train acc 0.8791
[save] epoch 1, acc 0.8791 -> best_ckpt.pt
epoch 2: train acc 0.8821
[save] epoch 2, acc 0.8821 -> best_ckpt.pt
epoch 3: train acc 0.8864
[save] epoch 3, acc 0.8864 -> best_ckpt.pt
epoch 4: train acc 0.8897
[save] epoch 4, acc 0.8897 -> best_ckpt.pt
epoch 5: train acc 0.8925
[save] epoch 5, acc 0.8925 -> best_ckpt.pt


要点：
- model.train() 与后面的 model.eval() 成对出现——忘写 train()，Dropout 层会在训练时乱丢神经元；忘写 eval()，推理时 Dropout 还在生效。zero_grad() 必须在 backward() 前——梯度默认累加，不清空会把上个 batch 的梯度叠进来。保存策略：只存最好的（if acc > best_acc），不是每 epoch 都存——中断/过拟合时能回滚到最佳版本。

correct += (pred.argmax(1) == y).sum().item()
- `pred.argmax(1)`：在第 1 维（10 个类别 logits）取最大值下标，得到预测类别，shape `[B]`
- `pred.argmax(1) == y`：布尔张量，预测对为 True，错为 False
- `.sum()`：统计 True 的数量，即本 batch 正确样本数
- `.item()`：把 tensor 数值取出变成 python 普通数字，累加到 correct

# 6.恢复并验证

In [23]:
ckpt = torch.load('best_ckpt.pt')
module2 = MLP()
module2.load_state_dict(ckpt['model_state_dict'])
print('恢复的 epoch:', ckpt['epoch'], ' |  当时的 acc:', ckpt['acc'])
print('权重张量数量:', len(ckpt['model_state_dict']))

恢复的 epoch: 5  |  当时的 acc: 0.89255
权重张量数量: 4


要点：
- 恢复是保存的逆操作：新建同结构模型 → load_state_dict 把权重"填"进去。注意 state_dict 不绑定类实例，只要求结构匹配（key 一一对应），这就是它能跨机器移植的原因。torch≥2.0 默认 weights_only=True（防反序列化攻击），存的是纯张量字典所以没影响

# 7.TensorBoard 记录

In [27]:
from torch.utils.tensorboard  import SummaryWriter
writer = SummaryWriter('runs/fashion_mlp')  # 日志目录 runs/实验名
global_step = 0        # 全局步数：代表已经训练过多少个 batch，不是 epoch
for epoch in range(3):
    module.train()
    for X, y in train_loader:
        pred = module(X)
        loss = loss_fn(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        global_step += 1
        if global_step % 50 == 0:          # 每 50 batch 记一次
            writer.add_scalar('train/loss', loss.item(), global_step)
            writer.add_scalar('train/acc_step', (pred.argmax(1) == y).float().mean().item(),global_step)
writer.close()
print('已写入 runs/fashion_mlp')

已写入 runs/fashion_mlp


要点
- （TensorBoard 的本质是插桩，不是自动）：writer.add_scalar(tag, value, step) 是手动一行行喂的。tag 用斜杠 'train/loss' 这种"域/指标"写法，TensorBoard 自动分组；step 用全局 batch 计数而不是 epoch——曲线才平滑。每 50 步记一次，别每步都写（日志文件会暴涨）。writer.close() 忘了可能丢尾部数据。
- 看曲线：WSL 终端对应内核环境中运行：
  - conda activate bl
  - cd /home/wsl2/py-learning-log/21-torch/Torch-learning-log
  - tensorboard --logdir=runs
  - 浏览器打开 http://localhost:6006。notebook 里跑不出界面——TensorBoard 是独立 Web 服务。


SummaryWriter 为 TensorBoard 里的日志写入工具，用来记录训练过程的曲线（loss、准确率等）

writer.add_scalar('train/loss', loss.item(), global_step)
- `add_scalar(标签名, 数值, x轴步数)`
  - `'train/loss'`：面板上曲线名字，`train`分组下的 loss 曲线；斜杠用来分组；
  - `loss.item()`：把 loss tensor 取出普通 Python 浮点数；
  - `global_step`：图表横轴。

# 8.AMP 自动混合精度

In [30]:
use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled = use_amp)
module = MLP()
optimizer = torch.optim.SGD(module.parameters(), lr = 0.1)
loss_fn = torch.nn.CrossEntropyLoss()
for epoch in range(3):
    module.train()
    for X, y in train_loader:
        optimizer.zero_grad()
        with torch.autocast(device_type = 'cuda', enabled = use_amp):
            pred = module(X)
            loss = loss_fn(pred, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    print(f'epoch {epoch}: loss {loss.item():.4f} (amp enabled: {use_amp})')

/tmp/ipykernel_3189/1264014720.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled = use_amp)


epoch 0: loss 0.4606 (amp enabled: True)
epoch 1: loss 0.5280 (amp enabled: True)
epoch 2: loss 0.3798 (amp enabled: True)


要点：
- AMP = FP16 算 + FP32 主权重。GPU 训练时，一部分计算用半精度`float16`，一部分用全精度`float32`；加速训练、减少显存占用。仅 NVIDIA CUDA GPU 生效，CPU/WSL 无 CUDA 时自动降级为 float32。显存减半、GPU 提速 1.5~3 倍。CPU 上无收益——你笔记本上跑这段只证明"代码会写"，数字没参考意义，加速要等实验室 GPU。
- **梯度缩放器 GradScaler**，AMP 配套必不可少。float16 数值范围很小，梯度很容易下溢变成 0；Scaler 作用：**把 loss 放大，让梯度变大避免下溢；更新的时候再缩放回来**。
- autocast 只包前向 + loss，不包 loss.backward()（反向由 scaler 管理）。
- 为什么需要 GradScaler：FP16 最大才 65504，最小规格化正数约 6e-5——反向传播里的梯度常比这还小，直接存 FP16 就下溢成 0，权重永远不更新。scaler 先把 loss 放大 2¹⁶ 倍再 backward，梯度跟着放大，step 前缩回去。
- `scaler.scale(loss)`：把 loss 乘上放大系数，专属于 AMP 的写法
- 三件套顺序固定：scaler.scale(loss).backward() → scaler.step(optimizer) → scaler.update()。

为什么这么写：
- enabled=use_amp 让 CPU 上这段和普通训练等价（不会报错、不会出假数字）——现在的机器跑它只是"练手写对结构"，等实验室 GPU 上把 use_amp 变 True 就自动加速。若以后遇到 torch 1.x，torch.autocast 要换成 torch.cuda.amp.autocast。

`scaler.step(optimizer)`：
- 内部做两件事：
1. 把放大的梯度**缩放还原回正确尺度**
2. 调用`optimizer.step()`更新权重

scaler.update()
- 根据是否出现 inf/nan 梯度，调整放大系数；如果出现非法梯度，会跳过本轮参数更新，动态修改缩放因子

AMP 固定执行顺序（必须严格遵守）
- 记忆口诀：**zero_grad → autocast 包裹前向 → scale.backward → scaler.step → update**

# 9.测评小结 

1.为什么只存 model.state_dict() 而不存整个 model？恢复训练为什么还要存optimizer.state_dict()？漏存会怎样？
- 以状态字典的形式保存不会出现移植时出现定义名一换就不匹配的问题，只存储其中的环节节点，保存框架；optimizer.state_dict 保存优化器内部缓存（Adam 动量、SGD 动量等）；漏存 optimizer 真正失去的是优化器内部状态（SGD 动量、Adam 的 m/v、步长缓存），那么优化器重新热身——收敛变慢、loss 曲线不连续

2.训练循环里忘写 optimizer.zero_grad() 会发生什么？忘写 model.train() / model.eval() 呢？
- 梯度越来越大，更新步长异常，loss 震荡、不下降、直接爆炸，训练完全失效；标准顺序：zero_grad() → 前向 → loss → backward() → step()；漏 model.train()：网络含 Dropout/BN 时，正则、BN 统计失效，训练效果差；漏 model.eval()：仍然处于训练模式；Dropout 继续随机失活，BN 用 batch 统计；预测结果不稳定、精度下降

3.add_scalar(tag, value, step) 三个参数各是什么？tag 写 'train/loss' 而不是 'loss' 有什么好处？step 为什么用全局 batch 数而不是 epoch 数？
- tag 是曲线的标签名，value 是曲线的数值点，step 表示图标的横轴；'train/loss' 中的 / 表示分组，这样更好管理；batch 数量更多，那么曲线相对 epoch 来说会更加的平滑；epoch 粒度太粗：3 个 epoch 只有 3 个点，曲线看不出训练过程（震荡、拐点全被平均掉了）；全局 step 还能让不同实验在同一横轴尺度上直接对比。平滑只是结果，分辨率才是原因。

4.AMP 里 autocast 为什么只包前向 + loss，不包 loss.backward()？GradScaler 具体解决什么问题（结合 FP16 的下溢说）？
-  `autocast` 只负责前向算子精度自动选择；反向传播复用前向计算图里已经确定的张量精度，不需要 autocast 管控，所以 backward 写在 with 块外面。 scaler 的解决机制：先把 loss 放大（典型 2¹⁶），梯度同比例放大进入 FP16 可表示范围，scaler.step() 内部 unscale 缩回，scaler.update() 动态调整缩放因子；深度学习反向传播产生很多微小梯度，直接用 FP16 存储会发生下溢，梯度变成 0，参数无法更新

5.你在 CPU 上跑 Cell 8，enabled=False 时这段代码实际在做什么？loss 数字有意义吗？什么时候它才有加速意义？
- `autocast`上下文失效，全部计算 FP32；有意义——enabled=False 时就是普通 FP32 训练，loss 是真实可靠的；它没意义的是"不能用来证明 AMP 有效"（没有加速、没有精度对比价值），`GradScaler`全部 API 退化为空操作，等价普通训练代码;只有真的在 GPU 上跑才有加速意义

In [32]:
import transformers

ModuleNotFoundError: No module named 'transformers'